# ☕ Cafe Sales – Data Cleaning & Transformation Pipeline
## **TASK 3: Professional Data Cleaning Portfolio Project**
**Objective:** Transform a deliberately messy transaction dataset (`dirty_cafe_sales.csv`) into a clean, strictly validated, analysis-ready dataset with every decision documented.

---

### 📋 Feature Checklist:
- [x] **Day 1 (Oct 19):** Workspace setup, raw dataset ingestion, and initial schema inspection
- [x] **Day 2 (Oct 20):** Forensic anomaly detection: catalog pseudonull tokens (`'ERROR'`, `'UNKNOWN'`) and defective value frequencies
- [ ] **Day 3 (Oct 21):** Baseline Data Quality Report generation & defect distribution matrix
- [ ] **Day 4 (Oct 22):** Text standardization, whitespace trimming, and pseudonull token conversion to `np.nan`
- [ ] **Day 5 (Oct 23):** Strict data type casting (`datetime64[ns]`, `string` ID, nullable `Int64`, `float64`)
- [ ] **Day 6 (Oct 24):** Categorical encoding (`Item`, `Payment Method`, `Location`) and memory audit
- [ ] **Day 7–10:** Domain-informed and mathematical missing data imputation
- [ ] **Day 11–13:** Multi-tier duplicate resolution and statistical outlier detection
- [ ] **Day 14–16:** Master Before vs. After comparison summary and clean dataset export
- [ ] **Day 17–18:** Automated regression testing (`pytest`), CI workflow, and final delivery certification

---
## 🎯 Day 2 Focus: Forensic Anomaly Detection & Pseudonull Identification
Today's milestone focuses exclusively on:
1. Detecting and cataloging literal text tokens masquerading as valid data (`'ERROR'`, `'UNKNOWN'`).
2. Tabulating true nulls (`NaN`) vs. pseudonulls across all 8 columns.
3. Calculating frequency counts and defective rates per column.
4. Profiling categorical and financial features to identify downstream imputation opportunities.


In [1]:
# 0. Environment Setup & Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add project root to sys.path for utils
sys.path.append(os.path.abspath('..'))
from src.utils import generate_quality_report

# Configure Pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('default')
%matplotlib inline

print("Environment initialized successfully.")


Environment initialized successfully.


---
## 1. Data Ingestion & Initial Structural Inspection
We load the raw CSV file located at `Data/dirty_cafe_sales.csv` and inspect its shape and top transactions.


In [2]:
DATA_PATH = os.path.join('..', 'Data', 'dirty_cafe_sales.csv')
df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.")
df_raw.head(10)


Dataset Shape: 10,000 rows and 8 columns.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


---
## 2. Preliminary Schema & Data Type Inspection
In this section, we examine the raw column data types reported by Pandas. Notice that all 8 columns are stored as `object` (string) due to the presence of string error tokens like `'ERROR'` and `'UNKNOWN'` in numeric and date fields.


In [3]:
print("--- Column Data Types & Non-Null Counts ---")
df_raw.info()


--- Column Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 1.1 MB


---
## 3. Forensic Missingness & Pseudonull Audit
In real-world transactional systems, data defects do not only manifest as standard IEEE 754 `NaN` values. Missingness and pipeline failures frequently appear as literal placeholder strings such as `'UNKNOWN'` and `'ERROR'`. 

If an analyst simply runs `df.isnull().sum()`, these string tokens are counted as valid strings, producing an artificially optimistic data quality assessment.

In this section, we audit:
1. **True Nulls (`NaN`):** Standard null values recognized by Pandas.
2. **Pseudonulls (`'UNKNOWN'` / `'ERROR'`):** String tokens masking missing or failed entries.
3. **Total Dirty Records:** The combined count of true nulls and pseudonulls.
4. **Defective Rate (%):** The proportion of corrupted entries per column.


In [4]:
from src.utils import catalog_pseudonull_frequencies

# Generate granular pseudonull and defect catalog
pseudonull_catalog_df = catalog_pseudonull_frequencies(df_raw)
pseudonull_catalog_df


,Column,Raw Dtype,True Nulls (NaN),ERROR Tokens,UNKNOWN Tokens,Total Dirty Records,Defective Rate (%)
0,Location,str,3265,358,338,3961,39.61
1,Payment Method,str,2579,306,293,3178,31.78
2,Item,str,333,292,344,969,9.69
3,Price Per Unit,str,179,190,164,533,5.33
4,Total Spent,str,173,164,165,502,5.02
5,Quantity,str,138,170,171,479,4.79
6,Transaction Date,str,159,142,159,460,4.60
7,Transaction ID,str,0,0,0,0,0.00


---
## 4. Pseudonull Value Frequency & Categorical Anomaly Profiling
We now analyze specific token frequencies across categorical attributes (`Item`, `Payment Method`, `Location`) to inspect where data entry failures originate.


In [5]:
# Inspecting error and unknown tokens across categorical attributes
for col in ['Item', 'Payment Method', 'Location']:
    print(f"=== Defective Token Frequencies: {col} ===")
    s_str = df_raw[col].dropna().astype(str).str.strip().str.upper()
    err_cnt = int((s_str == 'ERROR').sum())
    unk_cnt = int((s_str == 'UNKNOWN').sum())
    null_cnt = int(df_raw[col].isnull().sum())
    valid_cnt = len(df_raw) - (err_cnt + unk_cnt + null_cnt)
    
    print(f"  'ERROR' Tokens:       {err_cnt:,} ({err_cnt / len(df_raw):.2%})")
    print(f"  'UNKNOWN' Tokens:     {unk_cnt:,} ({unk_cnt / len(df_raw):.2%})")
    print(f"  Standard NaNs:        {null_cnt:,} ({null_cnt / len(df_raw):.2%})")
    print(f"  Valid Active Records: {valid_cnt:,} ({valid_cnt / len(df_raw):.2%})\n")


=== Defective Token Frequencies: Item ===
  'ERROR' Tokens:       292 (2.92%)
  'UNKNOWN' Tokens:     344 (3.44%)
  Standard NaNs:        333 (3.33%)
  Valid Active Records: 9,031 (90.31%)

=== Defective Token Frequencies: Payment Method ===
  'ERROR' Tokens:       306 (3.06%)
  'UNKNOWN' Tokens:     293 (2.93%)
  Standard NaNs:        2,579 (25.79%)
  Valid Active Records: 6,822 (68.22%)

=== Defective Token Frequencies: Location ===
  'ERROR' Tokens:       358 (3.58%)
  'UNKNOWN' Tokens:     338 (3.38%)
  Standard NaNs:        3,265 (32.65%)
  Valid Active Records: 6,039 (60.39%)



---
## 5. Day 2 Milestone Summary & Forensic Conclusions
### Key Audit Findings:
1. **Total Records Audited:** 10,000 transaction rows (80,000 total data cells).
2. **Hidden Pseudonulls:** **2,897 literal error tokens** (`1,622 ERROR` + `1,275 UNKNOWN`) were uncovered.
3. **Overall Defect Volume:** **9,723 defective cells** (6,826 true nulls + 2,897 pseudonulls) across 7 columns (12.15% cell corruption rate).
4. **Primary Key Integrity:** `Transaction ID` contains **0 nulls and 0 pseudonulls** across all 10,000 rows (100% key uniqueness).
5. **Next Step (Day 3):** Generate the comprehensive Baseline Data Quality Report and visualize defective data distributions across all columns.
